1.  TOTAL SALES: Total money collected from customers for all products sold.

In [0]:
kpi_1_df = spark.sql("""
    SELECT 
        ROUND(SUM(total_revenue), 2) AS Total_Sales 
    FROM de_mini_project.gold.fact_sales
    WHERE is_returned = 0
""")

display(kpi_1_df)

2.  PROFIT PER PRODUCT: Selling price minus the cost of the item.

In [0]:
kpi_2 = spark.sql("""
    SELECT 
        item_name, 
        category, 
        profit_per_unit AS Unit_Profit 
    FROM de_mini_project.gold.dim_product 
    ORDER BY profit_per_unit DESC
""")
display(kpi_2)

3.  TOP SELLING CATEGORY: Which group (e.g. Electronics) brings the most money.

In [0]:
kpi_3 = spark.sql("""
    SELECT 
        p.category, 
        ROUND(SUM(s.total_revenue), 2) AS Total_Revenue 
    FROM de_mini_project.gold.fact_sales s
    JOIN de_mini_project.gold.dim_product p ON s.product_id = p.product_id
    WHERE s.is_returned = 0
    GROUP BY p.category 
    ORDER BY Total_Revenue DESC 
    LIMIT 1
""")
display(kpi_3)

4.  BASKET SIZE: Average number of items bought per single transaction.

In [0]:
kpi_4 = spark.sql("""
    SELECT 
        ROUND(AVG(sold_qty), 2) AS Average_Items_Per_Basket 
    FROM de_mini_project.gold.fact_sales
""")
display(kpi_4)

5.  RETURN RATE: Percentage of total sales that were returned by customers.

In [0]:
kpi_5 = spark.sql("""
    SELECT 
        ROUND((SUM(is_returned) / COUNT(*)) * 100, 2) AS Return_Rate 
    FROM de_mini_project.gold.fact_sales
""")
display(kpi_5) 

6.  OUT-OF-STOCK COUNT: Count of unique products with zero units in stock.

In [0]:
kpi_6 = spark.sql("""
    SELECT 
        COUNT(DISTINCT product_id) AS Out_of_Stock_Count 
    FROM de_mini_project.gold.fact_inventory 
    WHERE is_out_of_stock = 1
""")
display(kpi_6)

7.  STORE PERFORMANCE: Ranking of the 5 stores based on daily sales volume.

In [0]:
kpi_7 = spark.sql("""
    SELECT 
        st.store_id,
        st.location, 
        st.region, 
        ROUND(SUM(sa.total_revenue), 2) AS Total_Sales 
    FROM de_mini_project.gold.fact_sales sa
    JOIN de_mini_project.gold.dim_store st ON sa.store_id = st.store_id
    WHERE sa.is_returned = 0
    GROUP BY st.store_id, st.location , st.region
    ORDER BY Total_Sales DESC
""")
display(kpi_7)

8.  DISCOUNT IMPACT: Money lost due to differences in marked vs. sold price.

In [0]:
kpi_8 = spark.sql("""
    SELECT 
        ROUND(SUM(discount_loss_amount), 2) AS Total_Discount_Loss 
    FROM de_mini_project.gold.fact_sales
    WHERE is_returned = 0
""")
display(kpi_8)

9.  REPEAT CUSTOMER COUNT: Number of unique users with more than one purchase.

In [0]:
kpi_9 = spark.sql("""
    SELECT 
        COUNT(*) AS Repeated_Customers_Count
    FROM (
        SELECT customer_id 
        FROM de_mini_project.gold.fact_sales 
        WHERE customer_id != 'UNKNOWN_CUSTOMER'
        GROUP BY customer_id 
        HAVING COUNT(DISTINCT transaction_id) > 1
    )
""")
display(kpi_9)

10. SLOW-MOVING INVENTORY: Products with zero sales recorded in the last 30 days.

In [0]:
kpi_10 = spark.sql("""
    SELECT 
        i.product_id,
        p.item_name,
        p.category,
        i.stocks_qty,
        i.last_sale_date
    FROM de_mini_project.gold.fact_inventory i
    JOIN de_mini_project.gold.dim_product p ON i.product_id = p.product_id
    WHERE i.is_slow_moving = 1
""")
display(kpi_10)